# Week 8: NLP Foundations & Text Embeddings
## Notebook 10: String Edit Distance: Levenshtein & Damerau-Levenshtein

### 1. Conceptual Intuition & Theoretical Background
Fuzzy string matching and spell correction require measuring the edit distance between strings.

**1. Levenshtein Distance (1965):**
Minimum number of single-character operations (**Insertion, Deletion, Substitution**) needed to transform source string $s$ into target $t$:
$$D(i, j) = \begin{cases} \max(i, j) & \text{if } \min(i, j) = 0 \\ \min \begin{cases} D(i-1, j) + 1 & (\text{Deletion}) \\ D(i, j-1) + 1 & (\text{Insertion}) \\ D(i-1, j-1) + \mathbb{I}(s_i \neq t_j) & (\text{Substitution}) \end{cases} & \text{otherwise} \end{cases}$$

**2. Damerau-Levenshtein Distance:**
Adds **Transposition** (swapping two adjacent characters, e.g. "teh" $\to$ "the"):
Counts as 1 edit rather than 2 substitutions, matching over $80\%$ of human typographical errors.

---

### 2. Scratch Python Implementation: Dynamic Programming Edit Matrix

In [ ]:
import numpy as np

def damerau_levenshtein_scratch(s, t):
    """
    Computes Damerau-Levenshtein distance with Transposition support.
    Operations: Insert, Delete, Substitute, Transpose.
    """
    len_s, len_t = len(s), len(t)
    dp = np.zeros((len_s + 1, len_t + 1), dtype=int)
    
    # Base cases
    for i in range(len_s + 1):
        dp[i, 0] = i
    for j in range(len_t + 1):
        dp[0, j] = j
        
    for i in range(1, len_s + 1):
        for j in range(1, len_t + 1):
            cost = 0 if s[i - 1] == t[j - 1] else 1
            
            # Standard Levenshtein operations
            dp[i, j] = min(
                dp[i - 1, j] + 1,      # Deletion
                dp[i, j - 1] + 1,      # Insertion
                dp[i - 1, j - 1] + cost # Substitution
            )
            
            # Damerau Transposition check
            if i > 1 and j > 1 and s[i - 1] == t[j - 2] and s[i - 2] == t[j - 1]:
                dp[i, j] = min(dp[i, j], dp[i - 2, j - 2] + 1)
                
    return dp[len_s, len_t]

# Comparison: Transposition Handling
word_a = "ca"
word_b = "ac"

# In Levenshtein: Substitution of 'c'->'a' + 'a'->'c' = 2 edits
# In Damerau-Levenshtein: Transposition of 'ca' -> 'ac' = 1 edit
dist_transposition = damerau_levenshtein_scratch("teh", "the")
dist_substitution = damerau_levenshtein_scratch("cat", "dog")

print("--- Dynamic Programming Edit Distance ---")
print(f"Edit Distance ('teh' -> 'the') [Typo Swap]: {dist_transposition} (Handled via Transposition)")
print(f"Edit Distance ('cat' -> 'dog') [Full change]: {dist_substitution}")
print(f"Edit Distance ('algorithm' -> 'altruistic'): {damerau_levenshtein_scratch('algorithm', 'altruistic')}")

### 3. Step-by-Step Code Walkthrough
1. **Dynamic Programming Table:** Builds an $(M+1) \times (N+1)$ table of subproblem solutions.
2. **Adjacent Transposition:** Checks if consecutive characters match swapped indices ($s[i-1] == t[j-2]$).
3. **Complexity:** $O(M \times N)$ time and space complexity.

---

### 4. Industry Applications & Trade-offs
- **Spell Checkers & Search Auto-Correction:** Suggesting corrected search queries (e.g., SymSpell).
- **Trade-off:** $O(M \times N)$ becomes slow for comparing genomic DNA sequences of millions of base pairs without Hirschberg's linear space algorithm.

---

### 5. Top Interview Questions & Answers
**Q1: How can you reduce the memory complexity of Levenshtein distance from $O(M \times N)$ to $O(N)$?**
> **Answer:** Computing row $i$ of the dynamic programming matrix only requires values from row $i-1$. By keeping only two rolling rows in memory, space complexity drops to $O(\min(M, N))$.

**Q2: What is the difference between Levenshtein distance and Hamming distance?**
> **Answer:** Hamming distance only allows substitutions and requires strings to be of equal length. Levenshtein distance allows insertions and deletions, handling strings of differing lengths.